# Implementation of fastText, BERT, and the phoneme LSTM model


*   fastText: breaks words into smaller character n-grams, allowing it to handle rare, misspelled, and out-of-vocabulary words better than traditional models like Word2Vec.
*   BERT: stands for Bidirectional Encoder Representations from Transformers. It understands context by analyzing words in both directions
* LSTM: is RNN designed to learn and remember long-term dependencies in sequential data, which is Japanese phonemes in this case.



## Load Dataset

### Dataset:
Load the `data/onomatopoeia.csv` dataset. The dataset compose of 220 unique onomatopoeia words, categorized as Movement, Sound, Visual, Texture, and Emotion. Those definitions are following:

*  Movement: Words describing physical motion or manner of movement
*  Sound: Words directly imitating auditory phenomena
* Visual: Words describing visual states or appearances
* Texture: Words describing tactile sensations, surface qualities, or material properties.
* Emotion: Words describing psychological states, feelings, or internal conditions.



In [ ]:
CSV_FILE = "data/onomatopoeia.csv"

colorMap = {
    "emotion": "royalblue",
    "movement": "orange",
    "sound": "forestgreen",
    "texture": "crimson",
    "visual": "mediumpurple"
}

## Install necessary libraries

In [ ]:
!pip install umap-learn japanize-matplotlib fasttext unidic_lite fugashi jaconv


## Import necessary libraries

In [ ]:
import os
import sys
import gzip
import shutil
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

import umap
from sklearn.manifold import TSNE
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.optim as optim
import fasttext
import jaconv
from transformers import BertJapaneseTokenizer, BertModel

import requests

## This is a common function that allows visualization of vectors from any model (FastText, BERT, LSTM) using UMAP

In [ ]:
def PlotUmap(xFeatures, yEncoded, labelEncoder, wordsList, title, fileName, colorMap):

    umapReducer = umap.UMAP(n_components=2, random_state=42)
    xUmap = umapReducer.fit_transform(xFeatures)

    plt.figure(figsize=(12, 10))

    for label in np.unique(yEncoded):
        labelIdx = (yEncoded == label)
        labelName = labelEncoder.classes_[label]

        plt.scatter(
            xUmap[labelIdx, 0],
            xUmap[labelIdx, 1],
            label=labelName,
            s=60,
            alpha=0.7,
            color=colorMap.get(labelName.lower(), "gray")
        )

    for i, word in enumerate(wordsList):
        plt.text(
            xUmap[i, 0],
            xUmap[i, 1],
            word,
            fontsize=8,
            alpha=0.8
        )

    plt.legend(title="Category")
    plt.title(title)
    plt.grid(True, alpha=0.3)

    plt.savefig(fileName, dpi=300, bbox_inches="tight")
    plt.show()

### Those two functions are to evaluate and compare the performance of ML classification models (Linear Regression and Linear SVM) using cross validation.

In [ ]:
def EvaluateDetailedResults(clf, xFeatures, yEncoded, labelEncoder, title):
    """
    It performs predictions for a single model (clf) and generates a detailed evaluation report for each class (category).
    """
    yPred = cross_val_predict(clf, xFeatures, yEncoded, cv=5) # cv=5 (cross validation)

    print(f"\n--- Detailed Classification Report: {title} ---")
    report = classification_report(
        yEncoded,
        yPred,
        target_names=labelEncoder.classes_,
        digits=4
    )

    print(report)

def RunEvaluation(lrClf, svmClf, xFeatures, yEncoded, labelEncoder, title):
    """
    It evaluates two different models (Logistic Regression (lrClf) and Linear SVM (svmClf) ) simultaneously
    and compare their results. It uses the Macro-F1 score as an evaluation metric.
    """
    print(f"\n{'='*30}")
    print(f" EVALUATION: {title} ")
    print(f"{'='*30}")

    scoresLr = cross_val_score(lrClf, xFeatures, yEncoded, cv=5, scoring="f1_macro")
    print(f"Logistic Regression (Mean Macro-F1): {scoresLr.mean():.4f}")
    EvaluateDetailedResults(lrClf, xFeatures, yEncoded, labelEncoder, f"{title} (LR)")

    scoresSvm = cross_val_score(svmClf, xFeatures, yEncoded, cv=5, scoring="f1_macro")
    print(f"Linear SVM          (Mean Macro-F1): {scoresSvm.mean():.4f}")
    EvaluateDetailedResults(svmClf, xFeatures, yEncoded, labelEncoder, f"{title} (SVM)")

    return scoresLr.mean(), scoresSvm.mean()

## Download the model for fastText
The pre-trained model `cc.ja.300.bin` is used.



In [ ]:
modelUrl = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ja.300.bin.gz"
gz = "cc.ja.300.bin.gz"
fastTextModel = "cc.ja.300.bin"

def DecomPressFile():
      try:
          with gzip.open(gz, 'rb') as f_in:
              with open(fastTextModel, 'wb') as f_out:
                  shutil.copyfileobj(f_in, f_out)
          print(f"Decompression complete.")
      except Exception as e:
          print(f"Error decompressing {gz}: {e}")
          if os.path.exists(fastTextModel):
              os.remove(fastTextModel)
          exit()

def DownLoadModel():
  if not os.path.exists(fastTextModel):
      if not os.path.exists(gz):
          try:
              response = requests.get(modelUrl, stream=True)
              response.raise_for_status()
              with open(gz, 'wb') as f:
                  for chunk in response.iter_content(chunk_size=8192):
                      f.write(chunk)

          except requests.exceptions.RequestException as e:
              print(f"Error downloading {gz}: {e}")
              exit()
      else:
          print(f"{gz} already exists, skipping download.")
      DecomPressFile()
  else:
      print(f"{fastTextModel} already exists, skipping download and decompression.")

DownLoadModel()
print("FastText model preparation step complete.")

## fastText Process

In [ ]:
def FastTextProcess(lrClf, svmClf, xFastText, yEncoded, labelEncoder):
    RunEvaluation(lrClf, svmClf, xFastText, yEncoded, labelEncoder, "fastText")

def Main():
    csvFile = CSV_FILE
    onomaDf = pd.read_csv(csvFile)

    labelEncoder = LabelEncoder()
    yEncoded = labelEncoder.fit_transform(onomaDf["category"])

    lrClf = LogisticRegression(max_iter=1000, random_state=42)
    svmClf = LinearSVC(max_iter=10000, random_state=42)

    fasttextModel = fasttext.load_model(fastTextModel)
    xFastText = np.array([fasttextModel.get_word_vector(w) for w in onomaDf["word"]])

    FastTextProcess(lrClf, svmClf, xFastText, yEncoded, labelEncoder)
    PlotUmap(
    xFastText, yEncoded, labelEncoder, onomaDf["word"].tolist(),
    "fastText", "umap_fasttext.png", colorMap
    )

if __name__ == "__main__":
    Main()

## Result of fastText:

| Category | Support | Logistic Regression (F1) | Linear SVM (F1) |
| :--- | :---: | :---: | :---: |
| **emotion** | 41 | 0.8941 | **0.8941** |
| **movement** | 55 | 0.8099 | **0.8571** |
| **sound** | 45 | 0.7470 | **0.7816** |
| **texture** | 45 | 0.8542 | **0.8842** |
| **visual** | 34 | 0.7273 | **0.7869** |
| **Overall Accuracy** | 220 | 0.8136 | **0.8455** |
| **Avg Macro-F1** | 220 | 0.8065  | **0.8408** |



# BERT
The pre-trained model `cl-tohoku/bert-base-japanese-v3` is used.

In [ ]:
modelName = "cl-tohoku/bert-base-japanese-v3"

def BertProcess(xBert, yEncoded, leBert):

    lrClf = LogisticRegression(random_state=42, max_iter=1000, multi_class='multinomial')
    svmClf = LinearSVC(random_state=42, max_iter=10000)

    meanLr, meanSvm = RunEvaluation(
        lrClf,
        svmClf,
        xBert,
        yEncoded,
        leBert,
        "BERT"
    )

    return meanSvm, meanLr

def FindSublistIndex(targetList, mainList):
    """
    It uses a sliding window to return the first starting index in `mainList` that
    exactly matches `targetList`. It returns -1 if no match is found.
    """
    targetLen = len(targetList)
    if targetLen == 0 or targetLen > len(mainList):
        return -1
    for i in range(len(mainList) - targetLen + 1):
        if mainList[i : i + targetLen] == targetList:
          return i

    return -1

def ExtractAndPoolEmbeddings(startIndex, wordLen, lastHiddenState):
    """
    It extracts the tensor within the specified range based on the start index,
    applies average pooling, and returns a vector (NumPy array).
    If nothing is found (startIndex == -1), it falls back to the [CLS] token.
    """

    if startIndex != -1:
        targetEmbeddings = lastHiddenState[startIndex : startIndex + wordLen, :]
        pooledVector = torch.mean(targetEmbeddings, dim=0).numpy()
        return pooledVector, False
    return lastHiddenState[0, :].numpy(), True

def GetTargetWordVector(targetWord, bertTokenizer, tokenInputs, lastHiddenState):
    """
    It identifies a set of tokens for the target word using the sliding window method
    and returns an Average Pooling vector.
    """
    wordTokens = bertTokenizer.encode(targetWord, add_special_tokens=False)
    sentenceTokens = tokenInputs['input_ids'][0].tolist()

    startIndex = FindSublistIndex(wordTokens, sentenceTokens)
    return ExtractAndPoolEmbeddings(startIndex, len(wordTokens), lastHiddenState)

def ExtractBertContextSpace(csvFilePath):
    onomaDf = pd.read_csv(csvFilePath)
    onomaDf['category'] = onomaDf['category'].str.strip()

    bertTokenizer = BertJapaneseTokenizer.from_pretrained(modelName)
    bertModel = BertModel.from_pretrained(modelName)
    bertModel.eval()

    bertEmbeddings = []
    plotLabels = []
    onomaCategories = []

    stats = {cat: {'total': 0, 'fallback': 0, 'fragmented': 0, 'has_unk': 0}
            for cat in onomaDf['category'].unique()}

    with torch.no_grad():
        for index, row in onomaDf.iterrows():
            inputSentence = str(row['sentence'])
            targetWord = str(row['word'])
            category = row['category']

            tokenInputs = bertTokenizer(
                inputSentence, return_tensors="pt", truncation=True, max_length=128, padding=False
            )
            modelOutputs = bertModel(**tokenInputs)
            lastHiddenState = modelOutputs.last_hidden_state[0]

            wordVector, isFallback = GetTargetWordVector(
                targetWord, bertTokenizer, tokenInputs, lastHiddenState
            )

            stats[category]['total'] += 1
            wordTokens = bertTokenizer.encode(targetWord, add_special_tokens=False)
            decoded_tokens = bertTokenizer.convert_ids_to_tokens(wordTokens)

            if isFallback:
                stats[category]['fallback'] += 1

            if '[UNK]' in decoded_tokens:
                stats[category]['has_unk'] += 1

            if len(wordTokens) >= 3:
                stats[category]['fragmented'] += 1

            bertEmbeddings.append(wordVector)
            plotLabels.append(targetWord)
            onomaCategories.append(category)

    print("-" * 40)
    print("BERT Tokenization Analysis per Category")
    for cat, data in stats.items():
        total = data['total']
        if total == 0: continue
        fallback_rate = (data['fallback'] / total) * 100
        unk_rate = (data['has_unk'] / total) * 100
        frag_rate = (data['fragmented'] / total) * 100

        print(f"Category: {cat} (Total: {total})")
        print(f"  - Fallback to [CLS]: {data['fallback']} ({fallback_rate:.1f}%)")
        print(f"  - Contains [UNK]:    {data['has_unk']} ({unk_rate:.1f}%)")
        print(f"  - Fragmented (>=3):  {data['fragmented']} ({frag_rate:.1f}%)")
    print("-" * 40)

    xBert = np.array(bertEmbeddings)
    return xBert, plotLabels, onomaCategories

if __name__ == "__main__":
    print("Extracting BERT Embeddings...")
    xBert, plotLabels, categories = ExtractBertContextSpace(CSV_FILE)

    print("Running Classification...")
    leBert = LabelEncoder()
    yEncoded = leBert.fit_transform(categories)
    BertProcess(xBert, yEncoded, leBert)

    print("Generating UMAP Plot...")
    PlotUmap(
      xBert, yEncoded, leBert, plotLabels,
      "BERT", "umap_bert.png", colorMap
    )

## Result of BERT Classification Performance

### Tokenization

| Category | Total | Fallback to [CLS] | Contains [UNK] | Fragmented (≥3) |
| :--- | :---: | :---: | :---: | :---: |
| **Sound** | 45 | 0 (0.0%) | 0 (0.0%) | 30 (66.7%) |
| **Movement** | 55 | 1 (1.8%) | 0 (0.0%) | 46 (83.6%) |
| **Texture** | 45 | 0 (0.0%) | 0 (0.0%) | 37 (82.2%) |
| **Visual** | 34 | 2 (5.9%) | 0 (0.0%) | 22 (64.7%) |
| **Emotion** | 41 | 2 (4.9%) | 0 (0.0%) | 26 (63.4%) |

### Classification Performance

| Category | Support | Logistic Regression (F1) | Linear SVM (F1) |
| :--- | :---: | :---: | :---: |
| **emotion** | 41 | 0.8500 | **0.8608** |
| **movement** | 55 | 0.8000 | **0.8037** |
| **sound** | 45 | 0.8090 | **0.8409** |
| **texture** | 45 | 0.8000 | **0.8298** |
| **visual** | 34 | 0.6761 | **0.6944** |
| **Overall Accuracy** | 220 | 0.7909 | **0.8091** |
| **Avg Macro-F1** | 220 | 0.7870 | **0.8059** |
